# Modélisation Fusion — Texte + Image
## Rakuten France — Classification Multimodale

**Objectif :** Combiner les prédictions du meilleur modèle texte et du meilleur modèle image  
pour dépasser les performances de chaque modalité prise séparément.

| Exp_ID | Stratégie | Description |
|--------|-----------|-------------|
| EXP_F01 | Moyenne simple | α fixe = 0.5 |
| EXP_F02 | Moyenne pondérée optimisée | α optimal trouvé par grid search sur val |
| EXP_F03 | Fusion adaptative | α varie selon `has_desc` et qualité image |
| EXP_F04 | Stacking | LogReg entraîné sur [proba_texte \| proba_image] |

**Hypothèse centrale :**  
Texte seul ≈ 0.82 · Image seule ≈ 0.65 · Fusion → **0.85+**

> Ce notebook nécessite d'avoir exécuté `modelisation_texte.ipynb` et `modelisation_images.ipynb`  
> (les probabilités sont recalculées ici depuis les modèles sauvegardés).

---
## 1. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import time
import warnings

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights, EfficientNet_B4_Weights
from PIL import Image

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 6)

BASE_PATH      = r'C:\Users\xavie\Desktop\Rakuten'
PROCESSED_PATH = os.path.join(BASE_PATH, 'processed')
MODELS_PATH    = os.path.join(BASE_PATH, 'models')
IMG_TRAIN_PATH = os.path.join(BASE_PATH, 'image_train')
IMG_TEST_PATH  = os.path.join(BASE_PATH, 'image_test')

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
SEED     = 42
N_CLASSES = 27

WHITE_THRESH = 240
BLUR_THRESH  = 100.0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device : {DEVICE}')
print('Imports OK')

---
## 2. Chargement des données

> **Important :** le split doit être identique aux notebooks texte et image  
> (`test_size=0.20, random_state=42, stratify=label`)

In [ ]:
train = pd.read_csv(os.path.join(PROCESSED_PATH, 'train_processed.csv'), index_col=0)
test  = pd.read_csv(os.path.join(PROCESSED_PATH, 'test_processed.csv'),  index_col=0)

with open(os.path.join(PROCESSED_PATH, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

for col in ['designation_clean', 'description_clean', 'text_combined']:
    train[col] = train[col].fillna('')
    if col in test.columns:
        test[col] = test[col].fillna('')

def get_img_path(row, folder):
    return os.path.join(folder, f"image_{row['imageid']}_product_{row['productid']}.jpg")

train['img_path'] = train.apply(lambda r: get_img_path(r, IMG_TRAIN_PATH), axis=1)
test['img_path']  = test.apply(lambda r:  get_img_path(r, IMG_TEST_PATH),  axis=1)

# Features qualité pour la fusion adaptative
train['has_desc'] = (train['description_clean'].str.strip() != '').astype(int)
test['has_desc']  = (test['description_clean'].str.strip() != '').astype(int)

# Split identique aux autres notebooks
X_tr, X_val, y_tr, y_val = train_test_split(
    train, train['label'],
    test_size=0.20, random_state=SEED, stratify=train['label']
)

classes = np.unique(y_tr)
class_weights = compute_class_weight('balanced', classes=classes, y=y_tr)
weight_map    = dict(zip(classes, class_weights))
sample_weights_tr = y_tr.map(weight_map).values

print(f'Train : {len(X_tr)} | Val : {len(X_val)}')
print(f'has_desc manquantes (val) : {(X_val["has_desc"]==0).sum()} ({(X_val["has_desc"]==0).mean():.1%})')

---
## 3. Probabilités texte

On ré-entraîne le meilleur pipeline texte (EXP_T03 : TF-IDF 50k + LightGBM) et on extrait  
les probabilités sur le jeu de validation et de test.  

> Si tu as identifié une meilleure config dans `modelisation_texte.ipynb`, adapte `BEST_TEXT_INPUT`.

In [ ]:
PROBA_TEXT_VAL_PATH  = os.path.join(PROCESSED_PATH, 'proba_text_val.npy')
PROBA_TEXT_TEST_PATH = os.path.join(PROCESSED_PATH, 'proba_text_test.npy')

# ⚠️ Adapter selon les résultats de modelisation_texte.ipynb
BEST_TEXT_INPUT = 'text_combined'  # ou 'designation_clean', 'text_lemmatized'...

if os.path.exists(PROBA_TEXT_VAL_PATH):
    print('Probabilités texte trouvées en cache — chargement...')
    proba_text_val  = np.load(PROBA_TEXT_VAL_PATH)
    proba_text_test = np.load(PROBA_TEXT_TEST_PATH)
    print(f'  val  : {proba_text_val.shape}')
    print(f'  test : {proba_text_test.shape}')
else:
    print(f'Entraînement TF-IDF + LightGBM (input: {BEST_TEXT_INPUT})...')
    t0 = time.time()

    tfidf = TfidfVectorizer(
        max_features=50_000, ngram_range=(1, 2),
        sublinear_tf=True, min_df=2, strip_accents='unicode'
    )
    X_tfidf_tr   = tfidf.fit_transform(X_tr[BEST_TEXT_INPUT])
    X_tfidf_val  = tfidf.transform(X_val[BEST_TEXT_INPUT])
    X_tfidf_test = tfidf.transform(test[BEST_TEXT_INPUT])

    lgb_tr  = lgb.Dataset(X_tfidf_tr,  label=y_tr, weight=sample_weights_tr)
    lgb_val_ds = lgb.Dataset(X_tfidf_val, label=y_val, reference=lgb_tr)

    params = {
        'objective': 'multiclass', 'num_class': N_CLASSES,
        'learning_rate': 0.05, 'num_leaves': 127,
        'seed': SEED, 'num_threads': -1, 'verbose': -1
    }
    model_text = lgb.train(
        params, lgb_tr, num_boost_round=1000,
        valid_sets=[lgb_val_ds],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(100)
        ]
    )

    proba_text_val  = model_text.predict(X_tfidf_val)   # (n_val, 27)
    proba_text_test = model_text.predict(X_tfidf_test)  # (n_test, 27)

    np.save(PROBA_TEXT_VAL_PATH,  proba_text_val)
    np.save(PROBA_TEXT_TEST_PATH, proba_text_test)

    wf1_text = f1_score(y_val, proba_text_val.argmax(axis=1), average='weighted')
    print(f'\nTexte seul — W-F1 = {wf1_text:.4f}  ({time.time()-t0:.0f}s)')
    print('Probabilités sauvegardées.')

wf1_text_only = f1_score(y_val, proba_text_val.argmax(axis=1), average='weighted')
print(f'\nRéférence texte seul — W-F1 = {wf1_text_only:.4f}')

---
## 4. Probabilités image

On charge le meilleur modèle image sauvegardé par `modelisation_images.ipynb`  
et on génère les probabilités sur validation et test.

> Adapter `BEST_IMG_CHECKPOINT` et `BEST_IMG_ARCH` selon vos résultats.

In [ ]:
# ⚠️ Adapter selon les résultats de modelisation_images.ipynb
BEST_IMG_CHECKPOINT = os.path.join(MODELS_PATH, 'EXP_I06_best.pth')  # ou EXP_I05, EXP_I04...
BEST_IMG_ARCH       = 'resnet50'     # 'resnet50' ou 'efficientnet_b4'
BEST_AUTOCROP       = True

# Fallback si EXP_I06 n'existe pas encore
if not os.path.exists(BEST_IMG_CHECKPOINT):
    for candidate in ['EXP_I05_best.pth', 'EXP_I04_best.pth',
                      'EXP_I02_best.pth', 'EXP_I01_best.pth']:
        path = os.path.join(MODELS_PATH, candidate)
        if os.path.exists(path):
            BEST_IMG_CHECKPOINT = path
            print(f'EXP_I06 non trouvé — fallback sur {candidate}')
            break

print(f'Checkpoint image : {os.path.basename(BEST_IMG_CHECKPOINT)}')
print(f'Architecture     : {BEST_IMG_ARCH}')
print(f'Auto-crop        : {BEST_AUTOCROP}')

In [ ]:
import cv2

PROBA_IMG_VAL_PATH  = os.path.join(PROCESSED_PATH, 'proba_image_val.npy')
PROBA_IMG_TEST_PATH = os.path.join(PROCESSED_PATH, 'proba_image_test.npy')
BLUR_SCORES_VAL_PATH = os.path.join(PROCESSED_PATH, 'blur_scores_val.npy')

def autocrop_white(img, white_thresh=WHITE_THRESH, border_pct=0.02):
    arr = np.array(img.convert('RGB'))
    non_white = ~np.all(arr >= white_thresh, axis=2)
    if not non_white.any():
        return img
    rows, cols = np.any(non_white, axis=1), np.any(non_white, axis=0)
    r0, r1 = np.where(rows)[0][[0, -1]]
    c0, c1 = np.where(cols)[0][[0, -1]]
    h, w = arr.shape[:2]
    b = int(min(h, w) * border_pct)
    return img.crop((max(0,c0-b), max(0,r0-b), min(w,c1+b+1), min(h,r1+b+1)))

def get_blur_score(img_path):
    try:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        return cv2.Laplacian(img, cv2.CV_64F).var() if img is not None else 0.0
    except:
        return 0.0

base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

class RakutenImgDataset(Dataset):
    def __init__(self, df, use_autocrop=False):
        self.df = df.reset_index(drop=True)
        self.use_autocrop = use_autocrop
        self.has_labels = 'label' in df.columns
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row['img_path']).convert('RGB')
            if self.use_autocrop:
                img = autocrop_white(img)
        except:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        t = base_transform(img)
        if self.has_labels:
            return t, int(row['label'])
        return t

def build_image_model(arch, checkpoint_path):
    if arch == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, N_CLASSES))
    elif arch == 'efficientnet_b4':
        from torchvision.models import efficientnet_b4
        m = efficientnet_b4(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.classifier[1].in_features, N_CLASSES))
    m.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
    return m.to(DEVICE).eval()

@torch.no_grad()
def get_probas(model, df, use_autocrop, batch_size=64):
    ds = RakutenImgDataset(df, use_autocrop)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    all_probas = []
    for batch in dl:
        imgs = batch[0] if isinstance(batch, (list, tuple)) else batch
        logits = model(imgs.to(DEVICE))
        all_probas.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(all_probas)

if os.path.exists(PROBA_IMG_VAL_PATH):
    print('Probabilités image trouvées en cache — chargement...')
    proba_img_val   = np.load(PROBA_IMG_VAL_PATH)
    proba_img_test  = np.load(PROBA_IMG_TEST_PATH)
    blur_scores_val = np.load(BLUR_SCORES_VAL_PATH)
else:
    print('Chargement modèle image...')
    model_img = build_image_model(BEST_IMG_ARCH, BEST_IMG_CHECKPOINT)

    print('Génération probabilités image (val)...')
    proba_img_val  = get_probas(model_img, X_val, BEST_AUTOCROP)
    print('Génération probabilités image (test)...')
    proba_img_test = get_probas(model_img, test,  BEST_AUTOCROP)

    print('Calcul blur scores (val)...')
    blur_scores_val = X_val['img_path'].apply(get_blur_score).values

    np.save(PROBA_IMG_VAL_PATH,   proba_img_val)
    np.save(PROBA_IMG_TEST_PATH,  proba_img_test)
    np.save(BLUR_SCORES_VAL_PATH, blur_scores_val)
    print('Probabilités sauvegardées.')

wf1_img_only = f1_score(y_val, proba_img_val.argmax(axis=1), average='weighted')
print(f'\nRéférence image seule — W-F1 = {wf1_img_only:.4f}')
print(f'Référence texte seul  — W-F1 = {wf1_text_only:.4f}')

---
## 5. Expériences de fusion

Dictionnaire global des résultats.

In [ ]:
FUS_RESULTS = {
    'TEXTE_SEUL': {
        'weighted_f1': wf1_text_only,
        'macro_f1': f1_score(y_val, proba_text_val.argmax(axis=1), average='macro'),
        'alpha': 1.0, 'description': 'Texte seul (référence)'
    },
    'IMAGE_SEULE': {
        'weighted_f1': wf1_img_only,
        'macro_f1': f1_score(y_val, proba_img_val.argmax(axis=1), average='macro'),
        'alpha': 0.0, 'description': 'Image seule (référence)'
    }
}

def fuse_and_eval(proba_t, proba_i, y_true, alpha):
    """Fusion pondérée : alpha * texte + (1-alpha) * image"""
    proba = alpha * proba_t + (1 - alpha) * proba_i
    preds = proba.argmax(axis=1)
    return (
        f1_score(y_true, preds, average='weighted'),
        f1_score(y_true, preds, average='macro'),
        preds
    )

print('Utilitaires de fusion définis.')
print(f'\nRéférences :')
print(f'  Texte seul  : W-F1 = {wf1_text_only:.4f}')
print(f'  Image seule : W-F1 = {wf1_img_only:.4f}')

---
## EXP_F01 — Moyenne simple (α = 0.5)

In [ ]:
wf1, mf1, preds = fuse_and_eval(proba_text_val, proba_img_val, y_val, alpha=0.5)
FUS_RESULTS['EXP_F01'] = {
    'weighted_f1': wf1, 'macro_f1': mf1,
    'alpha': 0.5, 'description': 'Moyenne simple (α=0.5)',
    'y_pred': preds
}
print(f'EXP_F01 — α=0.50 | W-F1={wf1:.4f} | M-F1={mf1:.4f}')
print(f'  Delta vs texte seul : {wf1 - wf1_text_only:+.4f}')

---
## EXP_F02 — Grid search sur α

On teste 50 valeurs d'α entre 0 et 1 pour trouver le poids optimal texte/image.  
La courbe montre comment les performances évoluent selon la pondération.

In [ ]:
alphas = np.linspace(0, 1, 51)
wf1_scores = []
mf1_scores = []

for a in alphas:
    wf1, mf1, _ = fuse_and_eval(proba_text_val, proba_img_val, y_val, alpha=a)
    wf1_scores.append(wf1)
    mf1_scores.append(mf1)

best_idx   = np.argmax(wf1_scores)
best_alpha = alphas[best_idx]
best_wf1   = wf1_scores[best_idx]
best_mf1   = mf1_scores[best_idx]

# Visualisation
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(alphas, wf1_scores, label='Weighted F1', color='steelblue', linewidth=2)
ax.plot(alphas, mf1_scores, label='Macro F1',    color='coral',     linewidth=2)
ax.axvline(best_alpha, color='green', linestyle='--',
           label=f'α optimal = {best_alpha:.2f} (W-F1={best_wf1:.4f})')
ax.axhline(wf1_text_only, color='gray', linestyle=':', alpha=0.7, label='Texte seul')
ax.axhline(wf1_img_only,  color='gray', linestyle='-.', alpha=0.7, label='Image seule')
ax.set_xlabel('α (poids du texte)')
ax.set_ylabel('F1 score')
ax.set_title('Grid search sur α — Weighted F1 en fonction du poids texte/image')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_PATH, 'docs', 'figures', 'fusion_alpha_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()

_, _, best_preds = fuse_and_eval(proba_text_val, proba_img_val, y_val, alpha=best_alpha)
FUS_RESULTS['EXP_F02'] = {
    'weighted_f1': best_wf1, 'macro_f1': best_mf1,
    'alpha': best_alpha, 'description': f'Alpha optimal (α={best_alpha:.2f})',
    'y_pred': best_preds
}
print(f'\nEXP_F02 — α optimal={best_alpha:.2f} | W-F1={best_wf1:.4f} | M-F1={best_mf1:.4f}')
print(f'  Delta vs texte seul : {best_wf1 - wf1_text_only:+.4f}')

---
## EXP_F03 — Fusion adaptative

Au lieu d'un α fixe pour tous les produits, on adapte le poids selon la **qualité des données disponibles** pour chaque produit :

| Condition | α (poids texte) | Raison |
|-----------|-----------------|--------|
| `has_desc=0` et image floue | 0.85 | Les deux signaux sont dégradés, texte reste dominant |
| `has_desc=0` | 0.80 | Texte pauvre → donner plus de place à l'image |
| Image floue | 0.75 | Image peu fiable → s'appuyer sur le texte |
| `has_desc=1` et image nette | α optimal (EXP_F02) | Les deux signaux sont fiables |

C'est l'hypothèse centrale du projet : la **complémentarité adaptative** des deux modalités.

In [ ]:
# Paramètres de la fusion adaptative — à ajuster selon vos résultats
ALPHA_BASE          = best_alpha         # α optimal trouvé en EXP_F02
ALPHA_NO_DESC       = 0.80               # has_desc = 0
ALPHA_BLURRY        = 0.75               # image floue
ALPHA_NO_DESC_BLURRY = 0.85             # les deux

has_desc_val    = X_val['has_desc'].values
is_blurry_val   = (blur_scores_val < BLUR_THRESH).astype(int)

# Vecteur d'alpha par produit
alpha_vec = np.where(
    (has_desc_val == 0) & (is_blurry_val == 1), ALPHA_NO_DESC_BLURRY,
    np.where(
        has_desc_val == 0, ALPHA_NO_DESC,
        np.where(is_blurry_val == 1, ALPHA_BLURRY, ALPHA_BASE)
    )
)

print('Distribution des alphas adaptatifs :')
for a_val, count in zip(*np.unique(alpha_vec, return_counts=True)):
    print(f'  α={a_val:.2f} : {count} produits ({count/len(alpha_vec):.1%})')

# Fusion avec alpha par ligne
proba_adaptive = alpha_vec[:, None] * proba_text_val + \
                 (1 - alpha_vec[:, None]) * proba_img_val
preds_adaptive = proba_adaptive.argmax(axis=1)
wf1_adaptive   = f1_score(y_val, preds_adaptive, average='weighted')
mf1_adaptive   = f1_score(y_val, preds_adaptive, average='macro')

FUS_RESULTS['EXP_F03'] = {
    'weighted_f1': wf1_adaptive, 'macro_f1': mf1_adaptive,
    'alpha': 'adaptatif', 'description': 'Fusion adaptative (has_desc + blur)',
    'y_pred': preds_adaptive, 'alpha_vec': alpha_vec
}
print(f'\nEXP_F03 — Fusion adaptative | W-F1={wf1_adaptive:.4f} | M-F1={mf1_adaptive:.4f}')
print(f'  Delta vs texte seul     : {wf1_adaptive - wf1_text_only:+.4f}')
print(f'  Delta vs alpha fixe     : {wf1_adaptive - best_wf1:+.4f}')

---
## EXP_F04 — Stacking (méta-apprentissage)

Au lieu de pondérer manuellement, on **apprend** comment combiner les deux modèles.  

Un méta-classifieur (LogisticRegression) est entraîné sur la concaténation des probabilités :  
`[proba_texte (27d) | proba_image (27d)] → 54 features → LogReg → classe finale`

Pour éviter le surapprentissage, les probabilités d'entraînement sont générées  
par **validation croisée 5 folds** (out-of-fold predictions).

In [ ]:
from sklearn.model_selection import StratifiedKFold

PROBA_TEXT_OOF_PATH = os.path.join(PROCESSED_PATH, 'proba_text_oof.npy')
PROBA_IMG_OOF_PATH  = os.path.join(PROCESSED_PATH, 'proba_img_oof.npy')

if os.path.exists(PROBA_TEXT_OOF_PATH) and os.path.exists(PROBA_IMG_OOF_PATH):
    print('OOF probas trouvées en cache — chargement...')
    proba_text_oof = np.load(PROBA_TEXT_OOF_PATH)
    proba_img_oof  = np.load(PROBA_IMG_OOF_PATH)
else:
    print('Génération des probabilités out-of-fold (5 folds)...')
    # On travaille uniquement sur X_tr pour le stacking
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    X_tr_reset = X_tr.reset_index(drop=True)
    y_tr_arr   = y_tr.values

    proba_text_oof = np.zeros((len(X_tr), N_CLASSES))
    proba_img_oof  = np.zeros((len(X_tr), N_CLASSES))

    model_img_oof = build_image_model(BEST_IMG_ARCH, BEST_IMG_CHECKPOINT)

    for fold, (idx_tr, idx_oof) in enumerate(skf.split(X_tr_reset, y_tr_arr), 1):
        print(f'  Fold {fold}/5...')

        # ── Texte OOF ──
        tfidf_oof = TfidfVectorizer(
            max_features=50_000, ngram_range=(1, 2),
            sublinear_tf=True, min_df=2, strip_accents='unicode'
        )
        X_fold_tr  = tfidf_oof.fit_transform(X_tr_reset.iloc[idx_tr][BEST_TEXT_INPUT])
        X_fold_oof = tfidf_oof.transform(X_tr_reset.iloc[idx_oof][BEST_TEXT_INPUT])
        y_fold_tr  = y_tr_arr[idx_tr]

        sw_fold = np.array([weight_map[y] for y in y_fold_tr])
        lgb_fold_tr  = lgb.Dataset(X_fold_tr,  label=y_fold_tr, weight=sw_fold)
        lgb_fold_oof = lgb.Dataset(X_fold_oof, label=y_tr_arr[idx_oof], reference=lgb_fold_tr)

        m = lgb.train(
            {'objective': 'multiclass', 'num_class': N_CLASSES,
             'learning_rate': 0.05, 'num_leaves': 127,
             'seed': SEED, 'num_threads': -1, 'verbose': -1},
            lgb_fold_tr, num_boost_round=500,
            valid_sets=[lgb_fold_oof],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)]
        )
        proba_text_oof[idx_oof] = m.predict(X_fold_oof)

        # ── Image OOF (même modèle, pas de re-train) ──
        proba_img_oof[idx_oof] = get_probas(
            model_img_oof, X_tr_reset.iloc[idx_oof], BEST_AUTOCROP
        )

    np.save(PROBA_TEXT_OOF_PATH, proba_text_oof)
    np.save(PROBA_IMG_OOF_PATH,  proba_img_oof)
    print('OOF probas sauvegardées.')

print(f'OOF shapes : texte={proba_text_oof.shape}, image={proba_img_oof.shape}')

In [ ]:
# Entraîner le méta-classifieur sur les OOF
X_meta_tr  = np.hstack([proba_text_oof, proba_img_oof])          # (n_train, 54)
X_meta_val = np.hstack([proba_text_val,  proba_img_val])          # (n_val, 54)

meta_clf = LogisticRegression(
    max_iter=1000, C=1.0, solver='lbfgs',
    multi_class='multinomial', random_state=SEED,
    class_weight='balanced'
)
meta_clf.fit(X_meta_tr, y_tr)

preds_stacking = meta_clf.predict(X_meta_val)
wf1_stacking   = f1_score(y_val, preds_stacking, average='weighted')
mf1_stacking   = f1_score(y_val, preds_stacking, average='macro')

FUS_RESULTS['EXP_F04'] = {
    'weighted_f1': wf1_stacking, 'macro_f1': mf1_stacking,
    'alpha': 'appris', 'description': 'Stacking (LogReg sur OOF probas)',
    'y_pred': preds_stacking
}
print(f'EXP_F04 — Stacking | W-F1={wf1_stacking:.4f} | M-F1={mf1_stacking:.4f}')
print(f'  Delta vs texte seul     : {wf1_stacking - wf1_text_only:+.4f}')
print(f'  Delta vs alpha fixe     : {wf1_stacking - best_wf1:+.4f}')
print(f'  Delta vs adaptatif      : {wf1_stacking - wf1_adaptive:+.4f}')

---
## 6. Comparaison et analyse

In [ ]:
rows = []
order = ['IMAGE_SEULE', 'TEXTE_SEUL', 'EXP_F01', 'EXP_F02', 'EXP_F03', 'EXP_F04']
for k in order:
    r = FUS_RESULTS[k]
    rows.append({
        'Exp_ID':      k,
        'Description': r['description'],
        'α':           r['alpha'],
        'Weighted F1': round(r['weighted_f1'], 4),
        'Macro F1':    round(r['macro_f1'], 4),
        'Delta texte': f"{r['weighted_f1'] - wf1_text_only:+.4f}"
    })

df_fus = pd.DataFrame(rows)
display(df_fus.style
    .background_gradient(subset=['Weighted F1', 'Macro F1'], cmap='RdYlGn')
    .format({'Weighted F1': '{:.4f}', 'Macro F1': '{:.4f}'})
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors  = ['#9E9E9E', '#9E9E9E', '#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
bars    = ax.barh(df_fus['Description'], df_fus['Weighted F1'], color=colors)
ax.axvline(wf1_text_only, color='gray', linestyle='--', alpha=0.8, label=f'Texte seul ({wf1_text_only:.4f})')
for bar, val in zip(bars, df_fus['Weighted F1']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_title('Comparaison stratégies de fusion — Weighted F1')
ax.set_xlabel('Weighted F1')
ax.legend()
ax.set_xlim(min(df_fus['Weighted F1']) - 0.05, max(df_fus['Weighted F1']) + 0.03)
plt.tight_layout()
os.makedirs(os.path.join(BASE_PATH, 'docs', 'figures'), exist_ok=True)
plt.savefig(os.path.join(BASE_PATH, 'docs', 'figures', 'fusion_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analyse par classe : gain de la meilleure fusion vs texte seul
best_fus_id = df_fus[~df_fus['Exp_ID'].isin(['TEXTE_SEUL','IMAGE_SEULE'])]\
    .sort_values('Weighted F1', ascending=False).iloc[0]['Exp_ID']

CLASS_NAMES = {
    0:'Livres', 1:'Jeux vidéo PC', 2:'Access. gaming', 3:'Consoles',
    4:'Dérivés geek', 5:'Cartes collec.', 6:'Figurines JS', 7:'Jouets',
    8:'Jeux société', 9:'Petites voitures', 10:'Access. enfance',
    11:'Jeux extérieur', 12:'Sacs/access.', 13:'Mobilier', 14:'Linge maison',
    15:'Nourriture', 16:'Lampes/déco', 17:'Access. animaux', 18:'Magazines',
    19:'Livres & BDs', 20:'Consoles/jeux', 21:'Papeterie', 22:'Mobilier ext.',
    23:'Piscines/Spas', 24:'Outillage', 25:'BDs & livres', 26:'Jeux DL'
}

f1_text = f1_score(y_val, proba_text_val.argmax(axis=1), average=None)
f1_fus  = f1_score(y_val, FUS_RESULTS[best_fus_id]['y_pred'],  average=None)
delta   = f1_fus - f1_text

df_delta = pd.DataFrame({
    'Classe':      [CLASS_NAMES[i] for i in range(N_CLASSES)],
    'F1 texte':    f1_text.round(3),
    f'F1 {best_fus_id}': f1_fus.round(3),
    'Delta':       delta.round(3)
}).sort_values('Delta', ascending=False)

fig, ax = plt.subplots(figsize=(10, 9))
colors_d = ['#4CAF50' if d > 0 else '#F44336' for d in df_delta['Delta']]
ax.barh(df_delta['Classe'], df_delta['Delta'], color=colors_d)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Gain de la fusion ({best_fus_id}) vs texte seul — par classe')
ax.set_xlabel('ΔF1 (vert = amélioration, rouge = dégradation)')
plt.tight_layout()
plt.savefig(os.path.join(BASE_PATH, 'docs', 'figures', 'fusion_delta_per_class.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('Classes les plus améliorées par la fusion :')
display(df_delta.head(5))
print('\nClasses dégradées par la fusion :')
display(df_delta[df_delta['Delta'] < 0].sort_values('Delta'))

---
## 7. Prédictions finales sur le jeu de test

In [ ]:
# Utiliser la meilleure stratégie de fusion sur le test
best_alpha_final = FUS_RESULTS[best_fus_id]['alpha']
print(f'Stratégie retenue : {best_fus_id} (α={best_alpha_final})')

if best_fus_id == 'EXP_F04':  # Stacking
    X_meta_test  = np.hstack([proba_text_test, proba_img_test])
    preds_test   = meta_clf.predict(X_meta_test)

elif best_fus_id == 'EXP_F03':  # Adaptatif
    has_desc_test  = test['has_desc'].values
    blur_scores_test = test['img_path'].apply(get_blur_score).values
    is_blurry_test   = (blur_scores_test < BLUR_THRESH).astype(int)
    alpha_test = np.where(
        (has_desc_test == 0) & (is_blurry_test == 1), ALPHA_NO_DESC_BLURRY,
        np.where(has_desc_test == 0, ALPHA_NO_DESC,
        np.where(is_blurry_test == 1, ALPHA_BLURRY, ALPHA_BASE))
    )
    proba_final = alpha_test[:, None] * proba_text_test + \
                  (1 - alpha_test[:, None]) * proba_img_test
    preds_test  = proba_final.argmax(axis=1)

else:  # Fixe (F01 ou F02)
    proba_final = best_alpha_final * proba_text_test + \
                  (1 - best_alpha_final) * proba_img_test
    preds_test  = proba_final.argmax(axis=1)

# Décoder les labels
prdtypecodes = le.inverse_transform(preds_test)

submission = pd.DataFrame({
    'Unnamed: 0':  test.index,
    'prdtypecode': prdtypecodes
}).set_index('Unnamed: 0')

submission_path = os.path.join(BASE_PATH, f'submission_{best_fus_id}.csv')
submission.to_csv(submission_path)
print(f'\nSubmission sauvegardée : {submission_path}')
print(f'Nombre de prédictions  : {len(submission)}')
display(submission.head())
print('\nDistribution des classes prédites :')
display(submission['prdtypecode'].value_counts().head(10))

In [ ]:
# Sauvegarde du tableau de résultats de fusion
df_fus.to_csv(os.path.join(PROCESSED_PATH, 'fusion_results.csv'), index=False)

print('=== RÉCAPITULATIF FINAL ===')
print(f'Image seule         : {wf1_img_only:.4f}')
print(f'Texte seul          : {wf1_text_only:.4f}')
for k in ['EXP_F01','EXP_F02','EXP_F03','EXP_F04']:
    if k in FUS_RESULTS:
        r = FUS_RESULTS[k]
        print(f'{k} ({r["description"][:30]}) : {r["weighted_f1"]:.4f}  ({r["weighted_f1"]-wf1_text_only:+.4f})')
print(f'\nBenchmark officiel : 0.8113 (text CNN) | 0.5534 (ResNet50)')

---
## Conclusions

*(À compléter après exécution)*

| Question | Résultat observé |
|----------|------------------|
| La fusion améliore-t-elle le texte seul ? | ? |
| Quelle stratégie est la meilleure ? | ? |
| La fusion adaptative bat-elle le fixe ? | ? |
| Quelles classes bénéficient le plus de l'image ? | ? |

**Prochain levier :** remplacer TF-IDF par SentenceTransformers/CamemBERT  
→ `modelisation_transformers.ipynb`  
Gain attendu sur le texte seul : **+0.05 à +0.08**, et donc sur la fusion : **+0.03 à +0.05** supplémentaires.